<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/miniprojects/MP7.1A/MP7.1A_slot_with_insulation.ipynb)

*Open this notebook in Google Colab with a GPU runtime (Runtime → Change runtime type → T4 GPU). Its first code cell fetches the library files from the public course repository.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Raissi, Perdikaris & Karniadakis, *Physics-informed neural networks*, J. Comput. Phys. 378 (2019) 686–707.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# MP7.1A · The Slot with Its Insulation

*Mini project of the course **Deep Learning for Engineering**, Part 2, in one notebook of five parts:*

**A** The Problem · **B** The Data and the Physics · **C** The Reference Model and the Ground Truth · **D** The Physics-Informed Network · **E** The Report

*Parts A to C are given and solved: read them, run them, and use what they build. Parts D and E are yours: the TODO cells.*

In [ ]:
# files-cell (mini project) ----------------------------------------------------
# The library files must sit beside the notebook. Locally they already do. On
# Google Colab they are fetched from the public course repository on every
# run, so a file left over from an earlier session cannot shadow it.
import os, sys, urllib.request
EXERCISE = (['course_core.py', 'pinn_core.py', 'problem.py'], "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex07.1-poisson/")
PROJECT = (['mp71a.py', 'MP7.1A_truth.npz'], "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/miniprojects/MP7.1A/")
ON_COLAB = "google.colab" in sys.modules
for files, url in (EXERCISE, PROJECT):
    for f in files:
        if ON_COLAB or not os.path.exists(f):
            if not ON_COLAB and os.path.exists(os.path.join("..", "..", "exercises", "Ex07.1-poisson", f)):
                import shutil; shutil.copy(os.path.join("..", "..", "exercises", "Ex07.1-poisson", f), f)
            else:
                urllib.request.urlretrieve(url + f, f)
            print("fetched", f)
for _m in ['course_core', 'pinn_core', 'problem', 'mp71a']:
    sys.modules.pop(_m, None)
print("files ready")

In [ ]:
# --- setup ---------------------------------------------------------------
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
import course_core as cc
import problem as hs
import mp71a as mp
torch.set_default_dtype(torch.float64)                   # a second derivative of a network needs double precision
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"  # the GPU, when the runtime has one
cc.OUTPUT_DIR = "MP7.1A_outputs"                          # where the report and its figures are written
print("device:", DEVICE, "| torch", torch.__version__)

def middle(fn, reps=5):
    """The middle of five wall-clock timings of fn(), after one warm-up run;
    on a GPU, synchronised so the time is the computation's, not the launch's."""
    sync = torch.cuda.synchronize if torch.cuda.is_available() else (lambda: None)
    fn(); sync(); ts = []
    for _ in range(reps):
        t0 = time.perf_counter(); fn(); sync(); ts.append(time.perf_counter() - t0)
    return float(np.median(ts))

pb = hs                                                  # Ex07.1's problem.py, by its name in Ex07.1

---

# Part A · The Problem

## A1 · What is computed, and why an engineer wants it

The winding of an electrical machine sits in slots cut into the stator iron. Its copper makes heat, $R I^2$, and that heat must cross the **insulation** - the enamel on every wire and the impregnating resin between the wires - to reach the iron, which the machine's cooling holds near 90 °C. The insulation is the slowest part of that path: its conductivity is 0.2 W/(m·K), two thousand times lower than copper's. The **hot spot**, the highest temperature in the slot, sets the life of the insulation: a common rule is that its life halves for every 8 to 10 K above its rating. A designer therefore wants the hot spot, and the field around it, for every current the machine will carry.

Ex07.1 answered that question for a slot whose winding was **averaged** into one material, $k_\mathrm{eff} = 0.70$ W/(m·K), with the copper's heat spread evenly. This mini project resolves every wire: the copper cores, their enamel coats and the resin between them. It asks how far the averaged answer is from the resolved one, and it builds a network that answers a **new current** in a fraction of the time a conventional solver needs.

In [ ]:
# The slot with every wire resolved, and an enlarged corner
fig, ax = plt.subplots(1, 2, figsize=(10, 6.4))
mp.draw_slot(ax[0])                                        # the whole slot, the quarter that is solved dashed
mp.draw_slot(ax[1], quarter=False, zoom=(2.4, 5.1, 7.4, 10.1))   # the top-right corner, enlarged
ax[0].legend(loc="upper center", bbox_to_anchor=(0.5, -0.09), fontsize=8, frameon=False)
ax[1].set_title("the corner: enamel 0.0375 mm thick, the gap to the wall 0.25 mm", fontsize=9)
plt.tight_layout(); plt.show()

rows = [("slot", f"{2*pb.A_HALF*1e3:.0f} × {2*pb.B_HALF*1e3:.0f} mm, walls held at the iron's {pb.T_WALL:.0f} °C"),
        ("winding", f"{pb.N_COLS} × {pb.N_ROWS} = {pb.N_WIRES} round wires, centres {mp.PITCH_X*1e3:.1f} mm apart both ways"),
        ("copper core", f"radius {mp.R_CU*1e3:.3f} mm, k = {mp.K_CU:.0f} W/(m·K)"),
        ("enamel (grade 2)", f"to radius {mp.R_ENAMEL*1e3:.4f} mm (IEC 60317), k = {mp.K_ENAMEL:.2f} W/(m·K)"),
        ("resin", f"everywhere else, k = {mp.K_RESIN:.2f} W/(m·K)"),
        ("current", f"{mp.I_MIN:.0f} to {mp.I_MAX:.0f} A per wire (RMS); rated {pb.I_RATED:.2f} A = 10 A/mm²"),
        ("copper's resistivity", f"{pb.RHO_CU:.2e} Ω·m at 20 °C, rising {pb.ALPHA_CU*100:.2f} % per kelvin")]
for k, v in rows:
    print(f"{k:>21}  {v}")

**What you should see.** Left, the slot of Ex07.1 drawn to scale: 32 copper cores (orange), each in a thin green enamel coat, in the beige resin, and the quarter the solvers use dashed. Right, the top-right corner enlarged: the enamel is only 0.0375 mm thick, and between the outer wires and the wall there is 0.25 mm of resin. Then the table of the data. Copper is Incropera & DeWitt's; the enamel's thickness is the IEC 60317 grade 2 minimum; **the enamel's and the resin's conductivity, 0.20 W/(m·K), are typical values and assumed** - the enamel has the same value as the resin, so the two insulators act as one material, but they are kept apart in the code so that either can be changed.

## A2 · What is new against Ex07.1

| | Ex07.1 | this mini project |
|---|---|---|
| the winding | one averaged material, $k_\mathrm{eff} = 0.70$ W/(m·K) | 32 copper cores, enamel and resin, each with its own $k$ |
| the heat | $q = 0.503\,\rho(T)\,J^2$, spread over the whole slot | $q = \rho(T)\,J^2$ in the copper only, nothing in the insulation |
| the conductivity | the same everywhere | jumps by a factor of 2000 at every copper edge |
| the solution | an exact double sine series | **none exists**: numerical only |

**Why no exact solution exists.** Ex07.1's series works because the slot is a rectangle with constant coefficients: every mode $\sin(m\pi x'/W)\sin(n\pi y'/H)$ solves the equation by itself. With 32 circles of copper inside, the coefficient $k(x, y)$ changes at every circle, no mode of the rectangle solves the equation, and no separation of variables or series exists. That is the normal case in engineering.

**The strength of the change.** In the solver every cell's conductivity and heat are mixed between the averaged slot ($S = 0$) and the resolved slot ($S = 1$). At $S = 0$ the solver must reproduce Ex07.1's exact series; part C checks it.

## A3 · The equation and its conditions

The unknown is the temperature rise above the wall, $\theta(x, y) = T - 90$ °C, which depends on the current $I$ in every wire. In steady state the heat made in a small area leaves it by conduction:

$$\nabla\cdot\big(k(x, y)\,\nabla\theta\big) + q(x, y, \theta) = 0 \quad\text{in the slot}, \qquad q = \begin{cases} c_0(I) + c_1(I)\,\theta & \text{in the copper} \\ 0 & \text{in the insulation} \end{cases}$$

with $c_0 = \rho(T_\mathrm{wall})\,J^2$ and $c_1 = \rho_{20}\,\alpha\,J^2$, $J = I/A_\mathrm{wire}$: the copper's resistance rises with its temperature, exactly as in Ex07.1, but now only in the copper.

| where | condition | why it holds |
|---|---|---|
| the four walls | $\theta = 0$ | the iron is cooled and conducts some hundred times better than the insulation; Ex07.1's condition |
| every copper edge, $r = 1.000$ mm from a centre | $\theta$ continuous; $k_\mathrm{cu}\,\partial\theta/\partial n = k_\mathrm{ins}\,\partial\theta/\partial n$ | no gap at the edge, so the temperature is the same on both sides, and no heat is stored there, so the heat flow leaving the copper enters the insulation |
| the lines $x = 0$ and $y = 0$ | $\partial\theta/\partial n = 0$ | the slot and its winding are symmetric about both: no heat crosses them, so a quarter is enough |

The middle row is the new one. The temperature is continuous but its **slope jumps** by the ratio of the conductivities: in the copper the slope is 2000 times smaller than just outside it. Each copper core is therefore almost at one temperature - part C measures 0.08 K across the hottest core at 45 A - and nearly all the temperature rise happens in the insulation.

In [ ]:
# The averaged answer of Ex07.1 against the resolved one, at rated current
G = np.load("MP7.1A_truth.npz")                            # the ground truth (part C explains how it is made)
i = int(np.argmin(np.abs(G["current"] - pb.I_RATED)))
avg = pb.exact_solution(pb.I_RATED, G["x"][::8], G["y"][::8])   # Ex07.1's exact series on the same quarter
fig, ax = plt.subplots(1, 2, figsize=(10, 6))
for a, f, t in ((ax[0], avg, "Ex07.1, averaged: exact series"), (ax[1], G["theta"][i][::8, ::8], "resolved: ground truth")):
    im = a.contourf(G["x"][::8] * 1e3, G["y"][::8] * 1e3, f, levels=np.linspace(0, 22, 23), cmap="inferno")
    a.set_aspect("equal"); a.set_title(t, fontsize=10); a.set_xlabel("x [mm]"); a.set_ylabel("y [mm]")
plt.colorbar(im, ax=ax, label="θ, rise above the wall [K]", shrink=0.8); plt.show()
print(f"hot spot at {pb.I_RATED:.2f} A: averaged {avg.max():.2f} K, resolved {G['hot_spot'][i]:.2f} K "
      f"({(G['hot_spot'][i] / avg.max() - 1) * 100:+.1f} %)")
print("the eight cores of the quarter [K]:", np.round(G["wire_theta"][i], 2))

**What you should see.** Two quarters of the slot, the centre of the slot in the bottom-left corner. The averaged field is a smooth dome; the resolved field is a set of plateaus - one per copper core - joined by steep slopes in the resin. The hot spot is 18.74 K averaged and **21.14 K resolved, 12.8 % higher**: the averaged $k_\mathrm{eff} = 0.70$ W/(m·K) is optimistic for this winding (the averaged slot gives the resolved hot spot with $k_\mathrm{eff} = 0.62$). The eight cores range from 21.14 K at the slot's centre to 4.74 K in the corner next to two walls.

## A4 · What is asked

| quantity | at | required |
|---|---|---|
| the hot spot, $\max\theta$, in the slot | the test currents 17.5, 31.42 and 41.5 A, which you do not train on | within **0.1 K** of the ground truth |
| the field $\theta(x, y)$ at the 17 824 scoring points of the quarter (part C) | the same currents | within **0.5 K** everywhere |
| the time to answer a new current online, the field at the scoring points | one Colab T4 runtime | at least **10 times shorter** than the reference model's (part C measures it) |

One network, trained once, offline, takes the current as an input: a new current is answered by evaluating it, not by training again. The training time is reported apart, as the offline cost.

**The agreed accuracy.** 0.1 K on the hot spot is 0.5 % of its value at rated current, a fifth of what the averaging of Ex07.1 costs, and smaller than any thermocouple could check. 0.5 K on the field is the accuracy the reference model's grid was chosen to meet (part C). Both the reference model and the solved network meet both.

---

# Part B · The Data and the Physics

## B1 · What the network takes and returns

$$(x,\ y,\ I)\ \longmapsto\ \theta(x, y; I)$$

the rise above the wall at a point of the quarter, for a current in every wire. The current is an **input**, like the coordinates: a new current is answered by evaluating the network.

**One network per material.** The conductivity jumps by a factor of 2000 at each copper edge, so the slope of $\theta$ jumps there too. A single smooth network cannot have a slope that jumps on 8 circles; it would round every corner. The usual remedy is **domain decomposition**: one network for the copper and one for the insulation, each smooth inside its own material, joined by the two conditions of the edge - the same temperature on both sides, and the same heat flow. The contract of part D asks for `net(x, y, I, part)`, with `part` either `"copper"` or `"resin"`, so that each material's network can be asked on its own and both can be asked on an edge.

The scales of this problem:

In [ ]:
# The scales: inputs and output of order one
I = pb.I_RATED
c0, c1 = mp.heat_coefficients(I)
print(f"the quarter: x from 0 to {pb.A_HALF*1e3:.0f} mm, y from 0 to {pb.B_HALF*1e3:.0f} mm; work in mm, or scale by the half-sizes")
print(f"the current: {mp.I_MIN:.0f} to {mp.I_MAX:.0f} A; the heat grows as I², so θ does too")
print(f"a scale of θ: Θ(I) = FILL c0 a² / (2 k_eff) = {mp.theta_scale(I):.2f} K at rated current (the hot spot is 21.14 K)")
print(f"the copper's heat at the wall temperature c0 = {c0/1e6:.2f} MW/m³, growing by c1 = {c1:.0f} W/(m³K) per kelvin ({c1/c0*100:.2f} %)")
print(f"the conductivities: copper {mp.K_CU:.0f}, insulation {mp.K_RESIN:.2f} W/(m·K), a ratio of {mp.K_CU/mp.K_RESIN:.0f}")

## B2 · The physics the network is trained with

Written in each material:

$$k_\mathrm{cu}\,\nabla^2\theta + c_0 + c_1\theta = 0 \ \text{ in the copper}, \qquad \nabla^2\theta = 0 \ \text{ in the insulation}$$

$$\theta_\mathrm{cu} = \theta_\mathrm{ins}, \qquad k_\mathrm{cu}\,\frac{\partial\theta_\mathrm{cu}}{\partial n} = k_\mathrm{ins}\,\frac{\partial\theta_\mathrm{ins}}{\partial n} \ \text{ on every copper edge}$$

$$\theta = 0 \ \text{ on the walls}, \qquad \frac{\partial\theta}{\partial n} = 0 \ \text{ on } x = 0 \text{ and } y = 0$$

| loss term | where it is evaluated | what it enforces |
|---|---|---|
| $\mathcal{L}_\mathrm{copper}$ | points inside the cores | the heat equation with its source |
| $\mathcal{L}_\mathrm{resin}$ | points in the insulation | Laplace's equation: no heat made there |
| $\mathcal{L}_\mathrm{T}$ | points on the copper edges | the same temperature from both networks |
| $\mathcal{L}_\mathrm{flux}$ | points on the copper edges | the same heat flow from both networks |
| $\mathcal{L}_\mathrm{wall}$ | points on the walls | $\theta = 0$ |
| $\mathcal{L}_\mathrm{sym}$ | points on the symmetry lines | no heat crosses them |

**Scale each term.** The residuals have different units and sizes: the copper's equation is of order $c_0 \approx 2$ MW/m³, Laplace's of order $\Theta/R^2$, the flux condition of order $k_\mathrm{ins}\Theta/R$. Divide each by its natural scale (part D's `losses` does), or one term will dominate the others by orders of magnitude - the lesson of MP8.2A.

**What can be built in rather than trained.** Every condition you build into the network's form holds for any weights and leaves less to train. Here: the walls ($\theta = 0$) hold if the output is multiplied by a function that vanishes on them, as in Ex07.1's hard enforcement; the symmetry lines hold if the network sees $x$ and $y$ only through even functions ($x^2$, $\cos(\pi x/a)$); and the equality of temperature on an edge holds if one network's value on the edge is given to the other. **The copper is nearly isothermal**: with $k_\mathrm{cu} = 400$ W/(m·K) its own heat raises its centre above its edge by only $c_0 R^2/(4 k_\mathrm{cu}) = 0.001$ K at rated current, so a core is well described by one temperature and a small dome - a network for it is almost not needed.

**The heat balance of each core.** Integrated over one core, the equation says that the heat it makes leaves through its edge:

$$\pi R^2\,\big(c_0 + c_1\bar\theta_j\big) = -\oint_{\text{edge } j} k_\mathrm{ins}\,\frac{\partial\theta_\mathrm{ins}}{\partial n}\,ds$$

where $n$ points out of the copper and $\bar\theta_j$ is the core's mean rise. Eight such balances, one per core, are a strong condition for a loss - and, written the other way round, a way to find the eight core temperatures for any current.

**A remark that changes the design.** The insulation makes no heat, so its field is **linear in the temperatures of the cores**: if every core were held at a given temperature, the insulation's field would be a sum of eight fixed fields, each multiplied by one core's temperature. Those eight fields do not depend on the current. Whether you use this is your choice in part D.

## B3 · The training points

`mp71a.py` provides samplers for every set of points the loss terms need, each with a current drawn uniformly from 10 to 45 A:

| sampler | returns | used for |
|---|---|---|
| `mp.sample_resin(n, seed, near, band)` | x, y [m] in the insulation, half of them within 0.3 mm of a copper edge | $\mathcal{L}_\mathrm{resin}$ |
| `mp.sample_copper(n, seed)` | x, y [m] inside the cores, uniform by area | $\mathcal{L}_\mathrm{copper}$ |
| `mp.sample_edges(n, seed)` | x, y [m] on the copper edges and the outward normal | $\mathcal{L}_\mathrm{T}$, $\mathcal{L}_\mathrm{flux}$ |
| `mp.sample_walls(n, seed)` | x, y [m] on $x = 5$ mm and $y = 10$ mm | $\mathcal{L}_\mathrm{wall}$ |
| `mp.sample_symmetry(n, seed)` | x, y [m] on $x = 0$ and $y = 0$ in the insulation, and the line's normal | $\mathcal{L}_\mathrm{sym}$ |
| `mp.resin_quadrature(nr, nt)` | Gauss points and weights covering the insulation exactly | integrals over the insulation |

The insulation is where the field bends: the half of the resin points placed close to the copper put them where the slopes are steepest. The quadrature covers each core's 2.5 mm square, minus the core, in polar coordinates about the core: its weights add up to the insulation's area, 24.87 mm², to rounding. An integral - a heat flow, an energy - is computed far more accurately with it than with random points.

The **test currents, 17.5, 31.42 and 41.5 A**, are inside the range but not on purpose in any sampler: the network is scored where it was not fitted.

In [ ]:
# The training points of the quarter
xr, yr, Ir = mp.sample_resin(3000)
xc, yc, Ic = mp.sample_copper(800)
xe, ye, nxe, nye, Ie = mp.sample_edges(400)
xw, yw, Iw = mp.sample_walls(200)
xs, ys, nxs, nys, Is = mp.sample_symmetry(200)
xq, yq, wq = mp.resin_quadrature(14, 14)
fig, ax = plt.subplots(1, 3, figsize=(13, 5.6), gridspec_kw={"width_ratios": [1, 1, 1.3]})
for a in ax[:2]:
    mp.draw_slot(a, quarter=False, zoom=(-0.3, 5.3, -0.3, 10.3))
ax[0].scatter(xr * 1e3, yr * 1e3, s=1, c="k", label="insulation")
ax[0].scatter(xc * 1e3, yc * 1e3, s=1, c="b", label="copper")
ax[0].scatter(xe * 1e3, ye * 1e3, s=3, c="m", label="edges")
ax[0].scatter(xw * 1e3, yw * 1e3, s=3, c="r", label="walls")
ax[0].scatter(xs * 1e3, ys * 1e3, s=3, c="g", label="symmetry lines")
ax[0].legend(fontsize=7, loc="upper center", bbox_to_anchor=(0.5, -0.1), ncol=2); ax[0].set_title("random points", fontsize=10)
ax[1].scatter(xq * 1e3, yq * 1e3, s=0.3, c="k"); ax[1].set_title(f"quadrature, {xq.size} points", fontsize=10)
ax[2].hist(Ir, bins=35); [ax[2].axvline(t, color="r") for t in mp.I_TEST]
ax[2].set_xlabel("current I [A]"); ax[2].set_ylabel("points"); ax[2].set_title("currents, test currents in red", fontsize=10)
plt.tight_layout(); plt.show()
print(f"quadrature: {xq.size} points, weights sum to {wq.sum()*1e6:.2f} mm² (insulation area {(50 - 8*np.pi):.2f} mm²)")

**What you should see.** Left, the quarter with its random points: dense rings of insulation points round each core, the cores filled with copper points, the edges, the walls and the symmetry lines. Middle, the quadrature: 12 544 points in rings round each core, graded towards the copper. Right, a flat distribution of currents from 10 to 45 A with the three test currents in red. The quadrature's weights sum to 24.87 mm², the insulation's area.

---

# Part C · The Reference Model and the Ground Truth

## C1 · Two solvers, two jobs

| | the **reference model** | the **ground truth** |
|---|---|---|
| what it is | the solver an engineer would run: finite volumes on an engineering grid | the same scheme on a grid fine enough to be ten to a hundred times more exact than anything scored |
| grid on the quarter | 60 × 120 cells of 0.083 mm | 400 × 800 cells of 0.0125 mm |
| its job | to be **timed** against the network: the online cost of a new current | to **score** the network and the reference |

Both solve the same discrete equations, written in `mp71a.Quarter`.

## C2 · The scheme, step by step

1. **Cells.** The quarter is cut into square cells of side $h$; the unknown is the mean rise in each cell.
2. **What each cell is made of.** In every cell the copper's edge crosses, the copper's share $f$ is counted on 8 × 8 points. The cell makes the heat $f\,(c_0 + c_1\theta)$ and conducts with $k = 1/(f/k_\mathrm{cu} + (1-f)/k_\mathrm{ins})$, the two materials in series: a cell that is half resin conducts about as badly as resin, which is how heat crosses a thin copper edge.
3. **Heat between neighbours.** The heat flowing from a cell to its neighbour is the difference of their rises times the conductance of the two half cells in series, the harmonic mean of their $k$. The heat flow is therefore the same leaving one cell and entering the other: across every material edge, the second edge condition holds by construction.
4. **The walls and the symmetry lines.** A cell on a wall loses heat to it through half a cell, $2k$ times its rise; on a symmetry line no heat crosses.
5. **One linear system per current.** The heat is affine in $\theta$, so each current is one sparse system, the conductance matrix minus the heat's growth on its diagonal. The matrix is assembled once for the grid; a new current is one sparse solve.

`mp.Quarter(h).solve(I)` returns the rise in every cell; `.balance` the heat made and the heat leaving through the walls.

In [ ]:
# The ground truth: its convergence at rated current, from the truth file
G = np.load("MP7.1A_truth.npz")
print("cell [mm]     cells    hot spot [K]   change of the field from the grid before [K]   energy balance off by")
for h, n, t, c, b in zip(G["conv_h_mm"], G["conv_cells"], G["conv_hot"], G["conv_field_change"], G["conv_balance"]):
    print(f"{h:8.4f}  {int(n):8d}    {t:10.4f}      {c:10.4f}                                    {abs(b):.1e}")
print(f"the ground truth's grid is 0.0125 mm; at S = 0 (one conductivity everywhere) the same scheme against "
      f"Ex07.1's exact series: {float(G['zero_strength_error_K']):.1e} K at most")
print(f"temperature spread inside the hottest core at 45 A: {float(G['copper_spread_K']):.2f} K")

**What you should see.** Five grids, each twice as fine. The hot spot changes by 0.096, 0.005, 0.0008 and 0.0014 K: it settles to about 0.001 K once the cells are 0.025 mm or finer. The field at the scoring points changes by 0.31, 0.010, 0.0027 and 0.0016 K. The ground truth's grid (0.0125 mm, the fourth) differs from the fifth by 0.0014 K on the hot spot and 0.0016 K on the field: seventy times finer than the 0.1 K asked and three hundred times finer than the 0.5 K. The order is not a clean two: the copper's edge cuts the cells at a different place on each grid, and the counted share $f$ changes irregularly, so a Richardson extrapolation would not be reliable here and is not used. The energy balance - the heat made equal to the heat leaving through the walls - closes to $10^{-9}$ on every grid. At $S = 0$ the same code reproduces Ex07.1's exact series to $2\times10^{-4}$ K: the code is right where the answer is known.

**Where the field is scored.** At 100 × 200 points 0.05 mm apart over the quarter, except the 2 176 that lie within 0.05 mm of a copper edge: there the slope jumps by 2000, and a value interpolated between two grid cells across the edge measures the interpolation, not the solution. 17 824 points remain. `mp.score_points(G)` returns them and the ground truth there.

In [ ]:
# The energy balance at every current of the truth file, and the test currents
print("current [A]   heat made [W/m]   heat through the walls [W/m]   hot spot [K]")
for I, b, h in zip(G["current"], G["balance"], G["hot_spot"]):
    mark = "   <- test current" if np.isclose(I, mp.I_TEST).any() else ""
    print(f"{I:9.2f}     {b[0]:11.2f}       {b[1]:11.2f}                  {h:8.2f}{mark}")

**What you should see.** Nine currents from 5 to 50 A, the test currents marked. The heat of the whole slot rises from 5.6 W per metre of slot at 5 A to 228.89 W/m at rated current and 616.5 W/m at 50 A - faster than $I^2$, because the copper's resistance grows with its temperature - and the walls take exactly what is made. The hot spot rises from 0.51 to 58.3 K. 5 and 50 A lie outside the training range, for question 9.

## C3 · The reference model: accuracy and time

The reference grid was chosen as the coarsest of a sequence (50, 60, 80, 100 cells across the quarter's 5 mm) that meets both agreed accuracies at the test currents: on 50 × 100 cells the hot spot is 0.19 K off and the field 0.56 K, on **60 × 120** 0.03 K and 0.30 K. The cell below scores it against the ground truth and times a new current: one sparse solve on the assembled grid, and the field read at the 17 824 scoring points. It runs on the CPU of the runtime: a sparse solve of this size gains nothing from a GPU.

In [ ]:
# The reference model against the ground truth, at the three test currents
xs, ys, truth = mp.score_points(G)                          # the scoring points [m] and the truth there [K]
grid = mp.Quarter(mp.REF_H)                                 # assembled once: the geometry does not change with I
worst_hot = worst_field = 0.0
for I in mp.I_TEST:
    i = int(np.argmin(np.abs(G["current"] - I)))
    field, hot = mp.reference_at(I, grid, xs, ys)
    e_hot, e_field = abs(hot - G["hot_spot"][i]), np.abs(field - truth[i]).max()
    worst_hot, worst_field = max(worst_hot, e_hot), max(worst_field, e_field)
    print(f"I = {I:5.2f} A: hot spot {hot:.2f} K (truth {G['hot_spot'][i]:.2f}), off by {e_hot:.3f} K; field off by {e_field:.3f} K at most")
t_ref = middle(lambda: mp.reference_at(30.0, grid, xs, ys))   # the online cost of a new current
print(f"\nreference model, {grid.nx} × {grid.ny} cells: worst hot spot {worst_hot:.3f} K, worst field {worst_field:.3f} K, "
      f"{t_ref*1e3:.1f} ms per new current")
print(f"the network must answer a new current in under {t_ref*1e3/10:.2f} ms (ten times shorter)")
REF = {"t_ref_s": t_ref, "worst_hot_K": worst_hot, "worst_field_K": worst_field}   # kept for part E

**What you should see.** At the three test currents the reference model's hot spot is within 0.03 K of the ground truth and its field within 0.30 K, inside the agreed 0.1 and 0.5 K. A new current takes some tens of milliseconds on a CPU (about 20 ms on the authors' laptop), almost all of it the sparse solve of 7 200 unknowns. The time is kept in `REF` for part E, which compares the network against it on this same runtime.

In [ ]:
# The ground truth at the three test currents, the target of part D
fig, ax = plt.subplots(1, 3, figsize=(11, 5.8))
for a, I in zip(ax, mp.I_TEST):
    i = int(np.argmin(np.abs(G["current"] - I)))
    im = a.contourf(G["x"][::4] * 1e3, G["y"][::4] * 1e3, G["theta"][i][::4, ::4], levels=24, cmap="inferno")
    a.set_aspect("equal"); a.set_title(f"{I:.2f} A, hot spot {G['hot_spot'][i]:.2f} K", fontsize=10); a.set_xlabel("x [mm]")
    plt.colorbar(im, ax=a, shrink=0.7, label="θ [K]")
ax[0].set_ylabel("y [mm]"); plt.tight_layout(); plt.show()

**What you should see.** The same pattern at three currents, scaled by about $I^2$: eight plateaus, the hottest core at the slot's centre (the bottom-left corner), the coolest in the top-right corner, the steepest slopes in the narrow gaps between the outer cores and the walls. Hot spots 6.33, 21.14 and 38.41 K.

---

# Part D · The Physics-Informed Network

## D1 · What is given, and what is yours

**Yours** are three things, each a TODO cell: the **network** (TODO 1), its **training** (TODO 2), and the **figures** you need to understand it (TODO 3).

**The contract.** Your network is a PyTorch module called `net`, used as `net(x, y, I, part)`:

| argument | meaning | shape |
|---|---|---|
| `x`, `y` | the point in the quarter, in metres: $0 \le x \le 5$ mm, $0 \le y \le 10$ mm | `(n, 1)` |
| `I` | the current in every wire, in amperes, 10 to 45 | `(n, 1)` |
| `part` | `"copper"` or `"resin"`: which material's network answers | a string |
| returns | the rise $\theta$ above the wall, in kelvin | `(n, 1)` |

The evaluation asks `part="copper"` at points inside a core and `part="resin"` elsewhere. Whatever you compute inside - features, built-in conditions, sub-networks, a solve - stays inside `net`.

## D2 · The loss, given

`losses(net, seed)` draws fresh points with the samplers of part B and returns the six residual terms of part B's table, each divided by its natural scale so that all are of order one: the equations by $c_0$ (copper) and by $\Theta/R^2$ (insulation), the conditions by $\Theta$ (temperatures) and by $k_\mathrm{ins}\Theta/R$ (heat flows), where $\Theta(I)$ is part B's scale of the rise and $R$ = 1 mm. Use the terms you need: a condition you build into `net` gives a zero term. `core_balance(net, I)` is the heat balance of the eight cores at the currents `I`, the heat made against the heat leaving each edge, computed with 64 points round each edge.

In [ ]:
# The loss, given ---------------------------------------------------------------
def T(a):
    """A NumPy array as an (n, 1) tensor on the device."""
    return torch.tensor(np.asarray(a, dtype=float), device=DEVICE).reshape(-1, 1)

def d(u, x):
    """du/dx by automatic differentiation, keeping the graph for a second derivative."""
    return torch.autograd.grad(u.sum(), x, create_graph=True)[0]

R0 = 1e-3                                                  # m, a length scale: the core's radius

def losses(net, seed, n=1500):
    def theta_scale(I):                                    # K, part B's scale Θ(I) as a tensor
        c0 = pb.resistivity(pb.T_WALL) * (I / pb.A_WIRE) ** 2
        return pb.FILL * c0 * pb.A_HALF ** 2 / (2 * pb.K_EFF)
    out = {}
    # the copper: k_cu ∇²θ + c0 + c1 θ = 0, divided by c0
    x, y, I = map(T, mp.sample_copper(n // 2, seed)); x.requires_grad_(True); y.requires_grad_(True)
    th = net(x, y, I, "copper")
    c0 = pb.resistivity(pb.T_WALL) * (I / pb.A_WIRE) ** 2; c1 = pb.RHO_CU * pb.ALPHA_CU * (I / pb.A_WIRE) ** 2
    out["copper"] = (((mp.K_CU * (d(d(th, x), x) + d(d(th, y), y)) + c0 + c1 * th) / c0) ** 2).mean()
    # the insulation: ∇²θ = 0, divided by Θ/R0²
    x, y, I = map(T, mp.sample_resin(n, seed + 1)); x.requires_grad_(True); y.requires_grad_(True)
    th = net(x, y, I, "resin")
    out["resin"] = (((d(d(th, x), x) + d(d(th, y), y)) * R0 ** 2 / theta_scale(I)) ** 2).mean()
    # the copper edges: the same temperature, and the same heat flow (n points out of the copper)
    x, y, nx, ny, I = map(T, mp.sample_edges(n // 3, seed + 2)); x.requires_grad_(True); y.requires_grad_(True)
    tc, tr = net(x, y, I, "copper"), net(x, y, I, "resin")
    out["T"] = (((tc - tr) / theta_scale(I)) ** 2).mean()
    qc = mp.K_CU * (d(tc, x) * nx + d(tc, y) * ny)         # heat flow leaving the copper [W/m²]
    qr = mp.K_RESIN * (d(tr, x) * nx + d(tr, y) * ny)      # heat flow entering the insulation
    out["flux"] = (((qc - qr) * R0 / (mp.K_RESIN * theta_scale(I))) ** 2).mean()
    # the walls: θ = 0
    x, y, I = map(T, mp.sample_walls(n // 5, seed + 3))
    out["wall"] = ((net(x, y, I, "resin") / theta_scale(I)) ** 2).mean()
    # the symmetry lines: no heat crosses
    x, y, nx, ny, I = map(T, mp.sample_symmetry(n // 5, seed + 4)); x.requires_grad_(True); y.requires_grad_(True)
    th = net(x, y, I, "resin")
    out["sym"] = (((d(th, x) * nx + d(th, y) * ny) * R0 / theta_scale(I)) ** 2).mean()
    return out

_ang = np.arange(64) * 2 * np.pi / 64                      # 64 points round each edge

def core_balance(net, I):
    """Per core and current: the heat made in it and the heat leaving its edge into the insulation [W/m],
    each (len(I), 8). They are equal when the solution is right."""
    made, left = [], []
    for Ii in np.atleast_1d(I):
        x = T(np.concatenate([cx + mp.R_CU * np.cos(_ang) for cx, cy in mp.QUARTER])).requires_grad_(True)
        y = T(np.concatenate([cy + mp.R_CU * np.sin(_ang) for cx, cy in mp.QUARTER])).requires_grad_(True)
        nx, ny = T(np.tile(np.cos(_ang), 8)), T(np.tile(np.sin(_ang), 8))
        It = T(np.full(x.shape[0], Ii))
        tr = net(x, y, It, "resin"); tc = net(x, y, It, "copper")
        flow = -(mp.K_RESIN * (d(tr, x) * nx + d(tr, y) * ny)).detach().cpu().numpy().reshape(8, 64)   # W/m², outward
        left.append(flow.mean(1) * 2 * np.pi * mp.R_CU)        # W/m: the mean flow times the edge's length
        c0, c1 = mp.heat_coefficients(Ii)
        made.append(np.pi * mp.R_CU ** 2 * (c0 + c1 * tc.detach().cpu().numpy().reshape(8, 64).mean(1)))
    return np.array(made), np.array(left)

## D3 · TODO 1 · Your network

Write the class `Net` and make `net = Net().to(DEVICE)`, honouring the contract above. Decide:

* **the decomposition.** One network per material, or one network with a part of its output for each; how the two are joined on the edges (both conditions in the loss, or one built in);
* **what is built in.** The walls, the symmetry lines, the copper's near-uniform temperature (part B, section 2), the growth of $\theta$ with $I^2$;
* **what the network sees.** $x$ and $y$ scaled, or even features of them; the current, scaled;
* **its size.** Layers, neurons, activation (tanh: the second derivatives must exist).

Two things the problem tells you. The insulation's field is linear in the core temperatures and makes no heat (part B). And the slope jumps by 2000 at the edges: a network asked to be smooth across them will not meet 0.5 K.

In [ ]:
# TODO 1 - your network. net(x, y, I, part) -> θ [K], x, y in metres and I in amperes, each (n, 1)
import torch.nn as nn

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        ...                                              # your networks, one per material, or your design

    def forward(self, x, y, I, part):
        ...                                              # return θ, shape (n, 1); part is "copper" or "resin"

torch.manual_seed(71)
net = Net().to(DEVICE)
print("weights:", sum(p.numel() for p in net.parameters()))

## D4 · TODO 2 · Your training

Train `net` with `losses` and, if you choose, `core_balance`. Decide the optimiser or optimisers, the steps, the learning rate and its schedule, the weight of each term, the points and whether they are drawn afresh. Keep the history of each term for TODO 3, and **measure the training time**: set `TRAIN_SECONDS`, which the evaluation reports as the offline cost.

Check the score as you go: `quick_score()` below gives the worst hot spot error and the worst field error over the three test currents, from the ground truth, in a second or two.

In [ ]:
# A quick score, given: the worst hot spot and field errors [K] at the test currents
G = np.load("MP7.1A_truth.npz")
xs, ys, truth = mp.score_points(G)                          # the 17 824 scoring points [m] and the truth there
CU_S = ~mp.in_resin(xs, ys)                                 # True for scoring points inside a copper core
X_CU, Y_CU, X_RS, Y_RS = T(xs[CU_S]), T(ys[CU_S]), T(xs[~CU_S]), T(ys[~CU_S])   # each material's scoring points
xh, yh = T(mp.QUARTER[:, 0]), T(mp.QUARTER[:, 1])           # the core centres, where the hot spot is

def answer(I):
    """The network's field at the scoring points and its hot spot [K], for one current I [A]: each material's
    network asked at its own points."""
    th = np.empty(len(xs))
    with torch.no_grad():
        th[CU_S] = net(X_CU, Y_CU, T(np.full(len(X_CU), I)), "copper").cpu().numpy().ravel()
        th[~CU_S] = net(X_RS, Y_RS, T(np.full(len(X_RS), I)), "resin").cpu().numpy().ravel()
        hot = net(xh, yh, T(np.full(8, I)), "copper").max()
    return th, float(hot)

def quick_score():
    worst_hot = worst_field = 0.0
    for I in mp.I_TEST:
        i = int(np.argmin(np.abs(G["current"] - I)))
        th, hot = answer(I)
        worst_hot = max(worst_hot, abs(hot - G["hot_spot"][i]))
        worst_field = max(worst_field, np.abs(th - truth[i]).max())
    return worst_hot, worst_field

In [ ]:
# TODO 2 - your training
history = {}
t0 = time.perf_counter()
...                                                      # your optimiser(s) and loop; append each term to history
TRAIN_SECONDS = time.perf_counter() - t0
print(f"trained in {TRAIN_SECONDS:.0f} s; worst errors: hot spot %.3f K, field %.3f K" % quick_score())

## D5 · TODO 3 · Your figures

Draw what you need to understand your network and to answer the report: at least the loss during training, and the field at rated current next to the ground truth, with its error. Save each figure you want in the report as `MP7.1A_report_<name>.png` (in `cc.output_path`): part E's PDF embeds them.

In [ ]:
# TODO 3 - your figures
...

## D6 · The evaluation, given

This cell asks your trained `net` the questions part E scores, and keeps the answers in the dictionary `R`:

* the field at the 17 824 scoring points and the hot spot at the three test currents, at the training range's ends (10 and 45 A) and outside it (5 and 50 A);
* the heat balance of the eight cores at rated current (`core_balance`);
* the **online time** of a new current on this runtime - the field at the scoring points and the hot spot, from one call of `net` per material - and the same for 10 and 100 currents, against the reference model run 1, 10 and 100 times on the same runtime.

Set `TRAIN_SECONDS` to the offline training time you measured; it is reported beside the online time.

In [ ]:
# The evaluation, given ------------------------------------------------------------
net.eval()
out = {"currents": np.array(G["current"]), "test": np.array(mp.I_TEST)}
fields, hots = [], []
for I in G["current"]:
    th, hot = answer(I)
    fields.append(th); hots.append(hot)
out["field"], out["hot"] = np.array(fields), np.array(hots)          # (9, 17824) and (9,) [K]
made, left = core_balance(net, [pb.I_RATED])
out["balance_made"], out["balance_left"] = made[0], left[0]           # W/m per core, rated current

# online time: a new current, the network against the reference model, on this runtime
grid = mp.Quarter(mp.REF_H)
timing = {}
for k in (1, 10, 100):
    Is = np.linspace(11.0, 44.0, k)
    timing[f"net_{k}"] = middle(lambda: [answer(I) for I in Is])
    timing[f"ref_{k}"] = middle(lambda: [mp.reference_at(I, grid, xs, ys) for I in Is], reps=3 if k == 100 else 5)
out.update({k: np.array(v) for k, v in timing.items()})
out["train_seconds"] = np.array(TRAIN_SECONDS)
out["n_weights"] = np.array(sum(p.numel() for p in net.parameters()))
out["device"] = np.array(DEVICE)
R = out                                                  # part E reads the answers from here
torch.save(net.state_dict(), cc.output_path("MP7.1A_net.pt"))   # the trained weights, beside the report
print(f"online, one new current: network {timing['net_1']*1e3:.2f} ms, reference {timing['ref_1']*1e3:.1f} ms "
      f"({timing['ref_1']/timing['net_1']:.0f} times)")
print("core heat balance at rated current [W/m], made:", np.round(made[0], 3))
print("                                       left:", np.round(left[0], 3))

---

# Part E · The Report

## E1 · What this part does

It scores your network's answers, kept by section D6 in `R`, against the ground truth `G` and the reference model's time `REF` from part C: the goal metrics (TODO 4) and the report's figures (TODO 5). Then come the ten questions.

In [ ]:
# What part E uses, all from the parts above
G = np.load("MP7.1A_truth.npz")
xs, ys, truth = mp.score_points(G)
print("in R:", ", ".join(R))
print(f"reference model (part C): worst hot spot {REF['worst_hot_K']:.3f} K, worst field {REF['worst_field_K']:.3f} K, "
      f"{REF['t_ref_s']*1e3:.1f} ms per new current")

## E2 · TODO 4 · The goal metrics

Compute, against the ground truth, and put in the dictionary `METRICS` (a name and a number each; the report prints them as its results table):

* the hot spot error and the largest field error at each test current (K);
* the network's online time for one new current, the reference model's on the same runtime (both in `R`), and their ratio;
* the offline training time.

Then say in one line each whether the three requirements of part A are met.

In [ ]:
# TODO 4 - the goal metrics
METRICS = {}
...
for k, v in METRICS.items():
    print(f"{k:>45}: {v:.3f}")

## E3 · TODO 5 · The report's figures

Draw, and save as `MP7.1A_report_<name>.png` in `cc.output_path`, the figures your answers refer to. At least: the hot spot against the current from 5 to 50 A, the network's and the ground truth's, with the error; the field error at the three test currents; and the online times of the network and the reference model for 1, 10 and 100 currents (`R["net_1"]`, `R["ref_10"]` and so on).

In [ ]:
# TODO 5 - the report's figures
...

## E4 · The report

Answer the ten questions below, each in a few sentences **with your numbers**; an answer needs at least fifteen words. Questions 1 to 5 are about your network and its result; 6 to 9 ask you to use it; 10 concludes. The next cell writes `MP7.1A_report.md` with the results table above and your answers, and the cell after it turns the report into a PDF. **Upload the PDF.**

In [ ]:
# TODO: write your answers. Every string below must be replaced.

NAME = "Your Name"

ANSWERS = {
    "1. Your network: how did you decompose it by material, how are the two parts joined on the copper edges, what does each see, and how many weights in all? Why this design?": """
""",
    "2. Which conditions did you build into the network's form, and which into the loss? Give each loss term's value at the end of training.": """
""",
    "3. Your training: optimiser(s), steps, learning rates, points or quadrature, and the offline time in seconds. Which term or quantity was hardest to bring down?": """
""",
    "4. The hot spot and the field at 17.5, 31.42 and 41.5 A: give each error in kelvin against the ground truth. Are all within 0.1 K and 0.5 K, and how do they compare with the reference model's?": """
""",
    "5. Where in the quarter is your largest field error at rated current, how large, and why there? Show it in a figure.": """
""",
    "6. The online time for a new current: your network against the reference model on the same runtime, in milliseconds, and the ratio. Then for 10 and 100 currents: what does each spend its time on?": """
""",
    "7. The heat balance of the eight cores at rated current: give the heat made and the heat leaving each core in W/m from your network. How far from equal are they, and what does that say about your field?": """
""",
    "8. How much does resolving the wires raise the hot spot against Ex07.1's averaged slot at 17.5, 31.42 and 41.5 A? Which k_eff would make the averaged slot give your resolved hot spot at rated current?": """
""",
    "9. Outside the training range: your hot spot errors at 5 A and 50 A. Does the error grow outside the range, and why, or why not, for your design?": """
""",
    "10. To conclude: for which engineering task would you use your network rather than the reference model, given your measured accuracy, online time and offline time? What would it take to answer a slot with another wire diameter or enamel grade?": """
""",
}

In [ ]:
MIN_WORDS = 15                                              # an answer is a few sentences with numbers, not a word
short = [(i, len(text.split())) for i, text in enumerate(ANSWERS.values(), 1) if len(text.split()) < MIN_WORDS]
if NAME.strip() in ("", "Your Name") or short:
    print("Not ready - the report is written once these are fixed:")
    if NAME.strip() in ("", "Your Name"):
        print("  NAME: write your name")
    for i, n in short:
        print(f"  answer {i}: {n} word(s); write at least {MIN_WORDS}")
else:

    rows = ["| metric | value |", "|---|---|"] + [f"| {k} | {v:.3f} |" for k, v in METRICS.items()]
    out = ["# MP7.1A — The Slot with Its Insulation", "", f"**{NAME}**", "", "Deep Learning for Engineering · Aalborg University · 2026", "",
           "## Results", "", *rows, ""]
    for qq, text in ANSWERS.items():
        out += [f"## {qq}", "", text.strip(), ""]
    path = cc.output_path("MP7.1A_report.md")
    with open(path, "w", encoding="utf-8") as fh:
        fh.write("\n".join(out))
    print("written", path)

**What you should see.** `Not ready` until every answer is yours; then the path of the report. Then run the cell below: it turns the report into a PDF, with the figures saved as `MP7.1A_report*.png`, and downloads it.

In [ ]:
# Report as PDF for Moodle -----------------------------------------------
# Runs after the report cell above: turns MP7.1A_report.md into MP7.1A_report.pdf, with any figure
# saved as MP7.1A_report*.png embedded above the answers, and downloads it. Upload
# the PDF to Moodle; the .md stays as the source.
import subprocess, sys, glob, os
pdf_path = os.path.join(cc.OUTPUT_DIR, "MP7.1A_report.pdf")
try:
    import markdown, weasyprint
except ImportError:                       # installed already on a second run
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "markdown", "weasyprint"])
    import markdown, weasyprint

md_path = os.path.join(cc.OUTPUT_DIR, "MP7.1A_report.md")
if not os.path.exists(md_path):
    print("No report yet: it is written once every answer above is filled in.\n"
          "Fill them in, run the cell that writes the report, then run this one again.")
else:
    md = open(md_path, encoding="utf-8").read()
    figs = sorted(glob.glob("MP7.1A_report*.png")
                  + glob.glob(os.path.join(cc.OUTPUT_DIR, "MP7.1A_report*.png")))
    if figs:
        imgs = "\n\n".join(f"![{os.path.basename(p)}]({p})" for p in figs)
        i = md.find("\n## ", md.find("## Results") + 1) if "## Results" in md else -1
        md = (md[:i] + "\n\n" + imgs + "\n" + md[i:]) if i > 0 else md + "\n\n" + imgs + "\n"

    html = markdown.markdown(md, extensions=["fenced_code", "tables"])
    css = """body{font-family:Helvetica,Arial,sans-serif;font-size:11pt;margin:2cm}
    h1{font-size:18pt} h2{font-size:13pt;margin-top:18pt}
    pre{background:#f3f4f6;padding:8px;font-size:9.5pt} img{max-width:100%}"""
    weasyprint.HTML(string=f"<html><head><meta charset='utf-8'><style>{css}</style></head>"
                           f"<body>{html}</body></html>", base_url=".").write_pdf(pdf_path)
    print("written", pdf_path, f"with {len(figs)} figure(s)" if figs else "")
    try:
        from google.colab import files
        files.download(pdf_path)
    except ImportError:
        pass